# Phase 2 — Experimental schema and quality control
## Open Materials Data Pipeline for Hydrogel Bioink Design

This notebook continues the completed **Phase 1 source-audit checkpoint** for Zenodo record [19602891](https://zenodo.org/records/19602891), version `v0.0.2`. It defines a source-grounded schema and exercises that schema against the supplied rheology archive and image index. It does not repeat acquisition, replace the original audit, or claim that records identify independent physical samples.

The main question is: **how can measured values retain their source location, units and experimental context when they are organised into reusable tables?** A schema is a contract describing what a row means, which fields it contains and which relationships may be relied upon. A useful materials schema must preserve both a number's computational representation and its physical meaning.

**Implemented scope:** checkpoint integrity checks; a documented entity model; field and unit mappings; a worksheet register, point register and source-cell table; image metadata with explicit missingness; executable structural and value checks; a strain-conversion preview; and a register of unresolved scientific questions. These are a validation prototype for Phase 2. Full standardisation, physical record linkage, curve fitting, formulation comparisons and machine learning remain later work. This bounded prototype exercises selected validation concepts from the README's conceptual Phase 3; it does not claim that phase is complete.

**Evidence levels used throughout**

- **Source-reported:** a value, header or method explicitly present in a supplied source.
- **Derived:** a deterministic software result, such as a cell coordinate or unit conversion.
- **Interpretation:** a reasoned reading of the available evidence, kept separate from a source assertion.
- **Unresolved:** not established by these inputs. Missing information is not replaced by a plausible guess.

No sample, formulation, printing-run or construct identifiers are manufactured. New field names below are explicitly **schema or provenance fields**, not claims that the source contains additional scientific measurements.

### Navigation

1. [Reproducible setup](#1.-Reproducible-setup-and-input-contract)
2. [Checkpoint verification](#2.-Verify-the-Phase-1-handoff)
3. [Literature evidence](#3.-Literature-evidence-and-its-scope)
4. [Entity and identifier design](#4.-Entities,-granularity-and-identifier-rules)
5. [Field and unit mapping](#5.-Source-to-schema-mapping-and-unit-policy)
6. [Source-preserving prototype](#6.-Instantiate-the-source-preserving-schema)
7. [Image metadata](#7.-Image-metadata-and-missing-value-semantics)
8. [Data dictionary](#8.-The-data-dictionary-as-an-executable-contract)
9. [QC rules and structural tests](#9.-Quality-control-rules-and-structural-validation)
10. [Measurement and image QC](#10.-Value-level-QC-without-silent-cleaning)
11. [Strain conversion preview](#11.-Traceable-strain-conversion-preview)
12. [Experimental context](#12.-Experimental-context-and-comparability)
13. [Issue decisions and linkage boundaries](#13.-Carry-forward-decisions-and-linkage-boundaries)
14. [Exports and provenance](#14.-Export-the-Phase-2-contract-and-handoff)

### Running this notebook

Place this file in `notebooks/`. Retain the two original source files in `data/raw/zenodo_19602891/` and all twelve Phase 1 outputs in `data/metadata/zenodo_source_audit/`, using their canonical filenames (without browser download suffixes). The notebook detects the project root from either the repository root or its `notebooks/` folder. If automatic detection fails, set `PROJECT_ROOT_OVERRIDE` in the first code cell to the local project folder.

Runtime dependencies: Python, pandas, NumPy, openpyxl and IPython in a Jupyter kernel. No network connection or API key is required. Exact tested versions are printed below and exported. Reading the supplied PDF and Word documents is not a runtime dependency: the reviewed evidence and document checksums are recorded explicitly in this notebook. Keep the papers and working documents for reference; do not assume the publication's licence applies to the dataset.

Use **Restart Kernel and Run All**. All variables are created here. Outputs are deterministically replaced only within `data/metadata/schema_and_quality_control/`; source files and Phase 1 outputs are opened for reading only. Timestamped run metadata will change between executions. The source-cell and image prototype exports are local review products, not an instruction to redistribute third-party data.


## 1. Reproducible setup and input contract

**Question.** Are the required source and checkpoint files available in the intended project layout?

**Data-science rationale.** A reproducible workflow declares its dependencies and file locations before processing. Relative paths make it portable; a fresh kernel exposes hidden dependencies on earlier interactive work. A checksum is a fingerprint of file bytes, not a judgement about scientific quality. Explicit failures are preferable to silently reading a similarly named or newer dataset.

**Polymer-science rationale.** The meaning of a rheology value depends on the measurement that produced it. Replacing a workbook or mixing audit versions could attach a property to the wrong conditions even if the code still runs. The snapshot, archive member and worksheet together provide the minimum source context.

**Output and boundary.** This cell reports the runtime and locates inputs. It does not establish that the source files are scientifically complete. The reviewed-document register records the files consulted during notebook construction; it is distinct from the runtime verification of data inputs.


In [3]:
from pathlib import Path, PurePosixPath
import csv, io, json, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version
import zipfile
import numpy as np
import pandas as pd
import openpyxl
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", None)

PROJECT_ROOT_OVERRIDE = None  # Optional local project directory; normally leave as None.
AUDIT_REL = Path("data/metadata/zenodo_source_audit")
RAW_REL = Path("data/raw/zenodo_19602891")
if PROJECT_ROOT_OVERRIDE is not None:
    project_root = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    project_root = next((p for p in candidates if (p / AUDIT_REL / "audit_run_manifest.json").is_file()), None)
    if project_root is None:
        raise FileNotFoundError("Cannot locate the Phase 1 audit. Place all twelve outputs in "
                                "data/metadata/zenodo_source_audit/ beneath the project root.")
audit_dir, raw_dir = project_root / AUDIT_REL, project_root / RAW_REL
output_dir = project_root / "data/metadata/schema_and_quality_control"
zip_path = raw_dir / "ALG-Ph_HA-Ph_rheology_data.zip"
image_path = raw_dir / "image_index.csv"
manifest_path = audit_dir / "audit_run_manifest.json"

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def require(condition, message):
    if not bool(condition):
        raise ValueError(message)

def is_blank(value):
    return value is None or (isinstance(value, str) and not value.strip())

for path in [zip_path, image_path, manifest_path]:
    if not path.is_file():
        raise FileNotFoundError(f"Required input missing: {path.relative_to(project_root)}")
environment = pd.DataFrame([{"component": "Python", "version": platform.python_version()}] +
                          [{"component": p, "version": version(p)} for p in ["pandas", "numpy", "openpyxl", "ipython"]])
display(environment)
print("Input layout located; no source files have been changed.")

reviewed_inputs = pd.DataFrame([{'reviewed_filename': '01_zenodo_source_audit(4).ipynb', 'sha256': 'e4de9e1602e54b71fa954c7480f12a233c9376d07d23208f70397df8590001bc', 'role': 'phase1_checkpoint'}, {'reviewed_filename': 'ALG-Ph_HA-Ph_rheology_data(3).zip', 'sha256': '8498a0a1c878bd1855d6767078f70b27e950b52b8b180201a46214063ed65ff4', 'role': 'data_or_audit'}, {'reviewed_filename': 'ChatGPT Prompt for Hydrogel Bioink(6).docx', 'sha256': 'a7de7e06fd5ffb3c9dee453739703b3b28fd4b6cb2509cd36b538a64c541c0a5', 'role': 'supporting_document'}, {'reviewed_filename': 'GENERA~1(1).DOC', 'sha256': '16bfe75b0ddf0be5029d384fed2f888ffdad9c0fe80e2803850d0b1cd2f98e81', 'role': 'supporting_document'}, {'reviewed_filename': 'Generative AI-guided in silico closed-loop optimisation of deposition morphology for 3D bioprinting applications(1).pdf', 'sha256': 'c3868eb63b4f4ce96be0b39ea4b058cc91ccf86d98697ff4ca673bfb560b6536', 'role': 'supporting_document'}, {'reviewed_filename': 'README_Phase1(1).txt', 'sha256': '9a4d8c75f5fa664bab50565e2d1472204cc00f9b86a57511a6fa6b4f875f74c3', 'role': 'supporting_document'}, {'reviewed_filename': 'archive_member_inventory(1).csv', 'sha256': 'a82ea7580b4941869ca0c1b2698efd50e7e4feec57678cfd0497237d0eb50751', 'role': 'data_or_audit'}, {'reviewed_filename': 'audit_issue_log(1).csv', 'sha256': '1b898b430040b1ea0e5cc76362923d3f7bc1d830e49d5eb60ef8223da4f6013e', 'role': 'data_or_audit'}, {'reviewed_filename': 'audit_run_manifest(1).json', 'sha256': 'a28b0638ef896d2c3ba04b65b8cd66f60a2d292f120b0eb28f91726876724193', 'role': 'data_or_audit'}, {'reviewed_filename': 'candidate_filename_correspondences(1).csv', 'sha256': 'a3f95d08e9393b7e871d9a2103b85f2c0306bb46357d36b8ac03cd6496e4003e', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_filename_expectations(1).csv', 'sha256': '6f0aff02ad2afb05ae39c53ff9da7313fbabd033f392253769d5d4acbb3ab759', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index(3).csv', 'sha256': '6e51d222d5434e271e4757ea750b8b4b002df13f056c21a5037d0e0b33a7b2c5', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index_column_profile(1).csv', 'sha256': '5885ff29927a70921059954aacacba7eba8ff33bcbad253337e99979a0459a60', 'role': 'data_or_audit'}, {'reviewed_filename': 'image_index_value_counts(1).csv', 'sha256': '40b1542dbf9d024e5db0d39b903035c4c1ad70a791704a2ed7a882ae2515dfd9', 'role': 'data_or_audit'}, {'reviewed_filename': 'measurement_axis_profile(1).csv', 'sha256': 'c0a30820727068b30ab3ca7d8a2aa49df353e5599982c2a5c07ddcf3c85bc40c', 'role': 'data_or_audit'}, {'reviewed_filename': 'measurement_column_profile(1).csv', 'sha256': '120aa04a12e56993d3821dc82be5569af30fe074039f509023f3fd05c236dca8', 'role': 'data_or_audit'}, {'reviewed_filename': 'source_header_inventory(1).csv', 'sha256': '8184bfff6a5d13b587044bfd0098f266c78542fb4aad30e6c1a1637d8f3b0ed7', 'role': 'data_or_audit'}, {'reviewed_filename': 'source_manifest(2).csv', 'sha256': '5fefea8090bbd598e12020f5b1be93eccde87a7a4b83faaa7836a5f54fa48be7', 'role': 'data_or_audit'}, {'reviewed_filename': 'workbook_structure_inventory(2).csv', 'sha256': 'e546d31e4f0434a9e797ee3555ca71b25afb653162dbacfc793a3ee813f1ee9a', 'role': 'data_or_audit'}])
display(reviewed_inputs[["reviewed_filename", "role"]])


,component,version
0,Python,3.12.15
1,pandas,2.2.3
2,numpy,1.26.4
3,openpyxl,3.1.5
4,ipython,9.17.1


Input layout located; no source files have been changed.


,reviewed_filename,role
0,01_zenodo_source_audit(4).ipynb,phase1_checkpoint
1,ALG-Ph_HA-Ph_rheology_data(3).zip,data_or_audit
2,ChatGPT Prompt for Hydrogel Bioink(6).docx,supporting_document
3,GENERA~1(1).DOC,supporting_document
4,Generative AI-guided in silico closed-loop optimisation of deposition morphology for 3D bioprinting applications(1).pdf,supporting_document
5,README_Phase1(1).txt,supporting_document
6,archive_member_inventory(1).csv,data_or_audit
7,audit_issue_log(1).csv,data_or_audit
8,audit_run_manifest(1).json,data_or_audit
9,candidate_filename_correspondences(1).csv,data_or_audit


**Observed output.** The project layout was found and all 19 reviewed inputs were registered. The validation environment used Python 3.12.14, pandas 2.2.3, NumPy 2.3.5, openpyxl 3.1.5 and IPython 9.17.1. These are the tested versions, not an assertion that every other version will fail. The next check verifies the checkpoint bytes.


## 2. Verify the Phase 1 handoff

**Question.** Do these bytes match the completed audit, and are all audit tables available?

**Data-science rationale.** The Phase 1 manifest links the two raw files and eleven CSV exports using SHA-256 hashes. Verifying those hashes establishes lineage: this schema is based on the reviewed snapshot. The manifest itself is pinned to the supplied checkpoint so a different manifest cannot silently redefine the input. All CSVs are first read as strings with automatic missing-value guessing disabled; an identifier or the word `NA` should not become a number or null by accident.

**Materials rationale.** The source has measurement points and metadata records, not a verified population of independent hydrogel samples. The manifest's counts are reconciliation targets rather than experimental sample sizes. Matching counts cannot resolve uncertainty about formulation identity or replication.

**Output and boundary.** The inventory shows every verified data input. A mismatch stops the notebook with an explanation. It requires a deliberate checkpoint review, not changing a hash just to make the test pass. This consumes the existing audit rather than rerunning its full profiling pipeline.


In [6]:
require(sha256(manifest_path) == 'a28b0638ef896d2c3ba04b65b8cd66f60a2d292f120b0eb28f91726876724193',
        "The Phase 1 manifest differs from the reviewed checkpoint. Review the changed audit before proceeding.")
phase1_manifest = json.loads(manifest_path.read_text(encoding="utf-8-sig"))
verified_paths = {manifest_path: sha256(manifest_path)}
verification_rows = []
for section, folder in [("input_sha256", raw_dir), ("export_sha256", audit_dir)]:
    for name, expected in phase1_manifest[section].items():
        path = folder / name
        if not path.is_file():
            raise FileNotFoundError(f"Missing checkpoint file: {path.relative_to(project_root)}")
        actual = sha256(path)
        require(actual == expected, f"Checksum mismatch: {name}. Restore the audited file or review the checkpoint.")
        verified_paths[path] = actual
        verification_rows.append(dict(relative_path=str(path.relative_to(project_root)), sha256=actual, status="matched"))
input_verification = pd.DataFrame(verification_rows)
audit_tables = {name: pd.read_csv(audit_dir / name, dtype=str, keep_default_na=False, encoding="utf-8-sig")
                for name in phase1_manifest["export_sha256"]}
workbook_inventory = audit_tables["workbook_structure_inventory.csv"]
column_profile = audit_tables["measurement_column_profile.csv"]
axis_profile = audit_tables["measurement_axis_profile.csv"]
phase1_issues = audit_tables["audit_issue_log.csv"]
require(len(audit_tables) == 11, "Expected eleven CSV exports from this checkpoint.")
display(input_verification[["relative_path", "status"]])
display(pd.DataFrame([{"quantity": k, "phase1_count": phase1_manifest[k]}
                     for k in ["xlsx_files_audited", "worksheets_audited", "measurement_rows", "image_index_rows", "issues"]]))


,relative_path,status
0,data\raw\zenodo_19602891\ALG-Ph_HA-Ph_rheology_data.zip,matched
1,data\raw\zenodo_19602891\image_index.csv,matched
2,data\metadata\zenodo_source_audit\source_manifest.csv,matched
3,data\metadata\zenodo_source_audit\archive_member_inventory.csv,matched
4,data\metadata\zenodo_source_audit\workbook_structure_inventory.csv,matched
5,data\metadata\zenodo_source_audit\source_header_inventory.csv,matched
6,data\metadata\zenodo_source_audit\measurement_column_profile.csv,matched
7,data\metadata\zenodo_source_audit\measurement_axis_profile.csv,matched
8,data\metadata\zenodo_source_audit\image_index_column_profile.csv,matched
9,data\metadata\zenodo_source_audit\image_index_value_counts.csv,matched


,quantity,phase1_count
0,xlsx_files_audited,92
1,worksheets_audited,92
2,measurement_rows,4600
3,image_index_rows,1031
4,issues,15


**Observed output.** Both raw files and all eleven audit CSVs matched their recorded SHA-256 values; the manifest matched the reviewed checkpoint. The handoff records 92 worksheets, 4,600 measurement rows, 1,031 image-index rows and 15 audit findings. These are record counts, not independent experimental sample sizes.


## 3. Literature evidence and its scope

The main publication is Zhang, Elvitigala and Sakai (2026), *Generative AI-guided in silico closed-loop optimization of deposition morphology for 3D bioprinting applications*, [DOI: 10.1080/17452759.2026.2671497](https://doi.org/10.1080/17452759.2026.2671497). The supplied supplementary document contains Figures S1–S15. Section and figure references below are more stable than PDF viewer page numbers. The README and working prompt define the project scope; they are not independent experimental evidence.

**Data-science rationale.** A literature evidence register stores a statement with its scope and its permitted use. Study-level methods should not be copied into every row as if independently measured there. This is the distinction between contextual metadata and record-level observation. The register contains paraphrases, not a reconstructed experimental dataset.

**Polymer-science rationale.** Alginate and hyaluronic acid are polymers. The publication describes their phenol-modified derivatives and a light-driven Ru/SPS system for crosslinking during printing. Polymer concentration, chain characteristics and phenol functionalisation can influence solution response and network formation. A hydrogel is a water-rich polymer network, but a rheology record must not automatically be labelled a cured gel: the paper distinguishes ink characterisation and printing/crosslinking operations. These source records do not provide a complete state history for every specimen.

Storage modulus G′ describes elastic energy storage; loss modulus G″ describes viscous dissipation under oscillatory deformation. A strain-amplitude sweep changes deformation magnitude, whereas a frequency sweep changes oscillation rate. These protocols answer different questions. The paper reports amplitude sweeps at 1 Hz; the actual workbook axes must still be checked. Its mention of the linear viscoelastic region does not identify a defensible plateau for every exported point.

**Output and boundary.** The evidence register improves interpretation while keeping unsupported row-level assignments unresolved. In particular, neither a study-wide reagent recipe nor a figure describing light settings establishes what a blank CSV annotation means.


In [9]:
evidence_register = pd.DataFrame([
    ["main_publication", "Abstract; Section 4", "The study concerns acellular hydrogel printing.", "study", "Do not claim measured cell viability or tissue function."],
    ["main_publication", "Section 5.1", "ALG-Ph and HA-Ph inks are described in PBS at pH 7.4; study concentration ranges are given in % (w/v).", "study", "Do not populate per-file concentrations or batch chemistry from an unverified filename convention."],
    ["main_publication", "Section 5.1", "Apparent viscosity is measured with a HAAKE MARS III rheometer and 40 mm parallel plates; the reported shear-rate range is 0.01–1000 s^-1 over 360 s.", "study method", "Compare with observed axes; do not discard lower-rate workbook points to enforce the prose range."],
    ["main_publication", "Section 5.1", "G-prime and G-double-prime are reported from oscillation-amplitude sweeps at 1 Hz, 0.5 mm gap and 22.5 degrees Celsius; the text refers to the LVR.", "study method", "Use as protocol context; retain observed strain and temperature. Do not certify every point as linear-viscoelastic."],
    ["main_publication", "Section 5.2", "The authors report 45 points for each fitted rheological parameter used in their SVR dataset.", "authors' modelling dataset", "Do not equate that modelling population with 46 source files per measurement family or force them to match."],
    ["main_publication", "Section 5.3", "Printing uses Ru/SPS-containing inks under 450 nm illumination; light settings span 0–100%, with a reported irradiance range of 0.001–13.571 W/m^2.", "study printing method", "Supports a percentage-setting interpretation, not a verified CSV field definition or a linear calibration."],
    ["supplementary_material", "Figure S3 caption", "Spectral irradiance is presented for the 450 nm light source at settings from 0 to 100%.", "figure / apparatus", "No digitisation or CSV-to-irradiance conversion is performed."],
    ["supplementary_material", "Figures S1 and S5 captions", "Blueprint designs and model-evaluation printability metrics are described.", "figure / authors' analysis", "Do not assign these metrics to image-index rows; the seven-column CSV does not contain those scores."],
    ["README_Phase1", "Phase 2; Current checkpoint", "Define entities, field mappings, units, missingness and evidence requirements from the completed audit.", "project scope", "Preserve unresolved links and distinguish this prototype from later standardisation and modelling."],
    ["working_prompt", "Source-first; experimental hierarchy; rerun requirements", "Preserve raw data and source lineage; explain science; use fresh-session execution and explicit uncertainty.", "project method", "Technical keys are source locators, not invented sample identifiers."]
], columns=["document", "source_location", "source_statement", "scope", "schema_decision"])
display(evidence_register)


,document,source_location,source_statement,scope,schema_decision
0,main_publication,Abstract; Section 4,The study concerns acellular hydrogel printing.,study,Do not claim measured cell viability or tissue function.
1,main_publication,Section 5.1,ALG-Ph and HA-Ph inks are described in PBS at pH 7.4; study concentration ranges are given in % (w/v).,study,Do not populate per-file concentrations or batch chemistry from an unverified filename convention.
2,main_publication,Section 5.1,Apparent viscosity is measured with a HAAKE MARS III rheometer and 40 mm parallel plates; the reported shear-rate range is 0.01–1000 s^-1 over 360 s.,study method,Compare with observed axes; do not discard lower-rate workbook points to enforce the prose range.
3,main_publication,Section 5.1,"G-prime and G-double-prime are reported from oscillation-amplitude sweeps at 1 Hz, 0.5 mm gap and 22.5 degrees Celsius; the text refers to the LVR.",study method,Use as protocol context; retain observed strain and temperature. Do not certify every point as linear-viscoelastic.
4,main_publication,Section 5.2,The authors report 45 points for each fitted rheological parameter used in their SVR dataset.,authors' modelling dataset,Do not equate that modelling population with 46 source files per measurement family or force them to match.
5,main_publication,Section 5.3,"Printing uses Ru/SPS-containing inks under 450 nm illumination; light settings span 0–100%, with a reported irradiance range of 0.001–13.571 W/m^2.",study printing method,"Supports a percentage-setting interpretation, not a verified CSV field definition or a linear calibration."
6,supplementary_material,Figure S3 caption,Spectral irradiance is presented for the 450 nm light source at settings from 0 to 100%.,figure / apparatus,No digitisation or CSV-to-irradiance conversion is performed.
7,supplementary_material,Figures S1 and S5 captions,Blueprint designs and model-evaluation printability metrics are described.,figure / authors' analysis,Do not assign these metrics to image-index rows; the seven-column CSV does not contain those scores.
8,README_Phase1,Phase 2; Current checkpoint,"Define entities, field mappings, units, missingness and evidence requirements from the completed audit.",project scope,Preserve unresolved links and distinguish this prototype from later standardisation and modelling.
9,working_prompt,Source-first; experimental hierarchy; rerun requirements,Preserve raw data and source lineage; explain science; use fresh-session execution and explicit uncertainty.,project method,"Technical keys are source locators, not invented sample identifiers."


**Observed output.** Ten evidence statements were recorded with explicit scope and schema decisions. The paper and supplementary caption support amplitude-sweep context and percentage light-setting context; neither establishes a physical link between an individual rheology file and an image. Source-coordinate structure can now be designed without adding such a link.


## 4. Entities, granularity and identifier rules

**Question.** What does each row represent, and which keys are justified?

**Data-science rationale.** *Granularity* is the level represented by a row. We separate a worksheet, a point within that worksheet, and a single value at that point. This avoids repeating file-level metadata in a wide table and gives each quality finding an exact source address. A **primary key** uniquely identifies a record within a table; a **foreign key** refers to an existing record in another table. Here they identify digital records, not physical objects.

The worksheet key is `(member_path, sheet)` within the verified archive snapshot. The point key adds `source_excel_row`; the cell key adds `source_column_position`. Archive SHA-256 is the enclosing namespace: if another snapshot is introduced, include its hash in cross-snapshot keys. The source `SegIndex` is retained but is not promoted to a global key. The image-record key is its CSV record number; `image_id` is a source label whose uniqueness is tested separately.

**Polymer-science rationale.** Many points on one sweep observe the same evolving acquisition, so 50 points cannot be counted as 50 independent materials. Conversely, two files with matching names need not use the same specimen. Molecular weight, concentration, crosslinking and processing history can affect properties even when a material label looks identical. No physical-sample table is populated until the source establishes sample identities.

**Output and boundary.** The entity map makes supported one-to-many source relationships explicit. Deferred entities are documented as evidence gaps rather than fabricated rows. A worksheet is called a measurement record rather than an independently verified experiment.


In [12]:
entity_schema = pd.DataFrame([
    ["worksheet_register", "one source worksheet", "member_path + sheet", "verified archive snapshot", "Digital measurement record; physical experimental identity unresolved"],
    ["measurement_points", "one nonblank source data row", "member_path + sheet + source_excel_row", "worksheet_register", "Repeated acquisition point; not an independent sample"],
    ["measurement_values", "one cell at a measurement point", "member_path + sheet + source_excel_row + source_column_position", "measurement_points", "Preserves both acquisition context and response values"],
    ["image_records", "one image-index CSV data record", "source_csv_record", "verified image-index snapshot", "Image metadata; no verified image, construct or printing-run identity"],
], columns=["table", "row_granularity", "primary_key", "parent_scope", "scientific_boundary"])
deferred_entities = pd.DataFrame([
    ["formulation", "Source labels retained; validated composition mapping is not yet established."],
    ["physical_sample", "No verified sample identifier or sample-to-measurement relationship."],
    ["printing_event", "No verified run identifier linking rheology and image records."],
    ["printed_construct", "Image IDs do not establish construct identity or repeat imaging."],
], columns=["entity", "reason_not_instantiated"])
display(entity_schema)
display(deferred_entities)


,table,row_granularity,primary_key,parent_scope,scientific_boundary
0,worksheet_register,one source worksheet,member_path + sheet,verified archive snapshot,Digital measurement record; physical experimental identity unresolved
1,measurement_points,one nonblank source data row,member_path + sheet + source_excel_row,worksheet_register,Repeated acquisition point; not an independent sample
2,measurement_values,one cell at a measurement point,member_path + sheet + source_excel_row + source_column_position,measurement_points,Preserves both acquisition context and response values
3,image_records,one image-index CSV data record,source_csv_record,verified image-index snapshot,"Image metadata; no verified image, construct or printing-run identity"


,entity,reason_not_instantiated
0,formulation,Source labels retained; validated composition mapping is not yet established.
1,physical_sample,No verified sample identifier or sample-to-measurement relationship.
2,printing_event,No verified run identifier linking rheology and image records.
3,printed_construct,Image IDs do not establish construct identity or repeat imaging.


**Observed output.** Four source-grounded entities were defined. Four physical or process entities remain deferred because their identities or relationships are not established. The hierarchy supports tracing a cell to its source worksheet, but it cannot answer which physical sample was used across different files.


## 5. Source-to-schema mapping and unit policy

**Question.** Which physical quantities do the observed headers represent, and which transformations are defensible?

**Data-science rationale.** A source-to-schema map pairs an exact source header with a proposed canonical field, type, unit and transformation policy. Schema drift means a source's representation varies across files; the two strain headers are a concrete example. The code requires coverage of all observed headers, so an unexpected new header cannot silently disappear.

**Rheology rationale.** Shear stress (Pa) and shear rate (s⁻¹) are not viscosity. Apparent steady-shear viscosity describes resistance to flow at a specified shear rate. Complex viscosity belongs to oscillatory testing; it must not be substituted for steady-shear viscosity merely because both use viscosity units. Temperature and time are conditions or acquisition coordinates, not predictors automatically ready for modelling. The source's `t_seg` and `SegIndex` tokens are preserved; detailed instrument segmentation semantics remain undocumented here.

Strain is dimensionless. If the source reports percent strain, the corresponding fraction is `value / 100`; a fraction of 0.01 equals 1%. This mathematical rule does not locate the LVR. `mPas` is preserved exactly as an unresolved source token in the production mapping. If established as mPa·s, its factor to Pa·s would be 0.001, but this notebook does not treat that conditional interpretation as a confirmed conversion. Frequency remains Hz; no angular-frequency column is added because this schema test does not require it.

**Output and boundary.** The mappings are new schema definitions, not new observations. Type parsing and a separate strain preview are tested below. No normalised rheology dataset is labelled complete, and unresolved viscosity or intensity units are not guessed.


In [15]:
field_mapping = pd.DataFrame([
    ["SegIndex", "segment_index_source", "string", "not applicable", "retain", "source acquisition label; detailed segmentation semantics unresolved"],
    ["t in s", "time_s", "number", "s", "identity", "source time coordinate"],
    ["t_seg in s", "segment_time_s", "number", "s", "identity", "source segment-time coordinate; zero-time convention unresolved"],
    ["τ in Pa", "shear_stress_pa", "number", "Pa", "identity", "stress under the recorded test mode"],
    ["G' in Pa", "storage_modulus_pa", "number", "Pa", "identity", "elastic storage contribution under recorded oscillation conditions"],
    ['G" in Pa', "loss_modulus_pa", "number", "Pa", "identity", "viscous loss contribution under recorded oscillation conditions"],
    ["|η*| in mPas", "complex_viscosity_source", "number", "mPas", "deferred", "oscillatory complex viscosity; source unit token retained"],
    ["ɣ in -", "strain_fraction", "number", "-", "identity", "dimensionless strain fraction as labelled"],
    ["ɣ in %", "strain_fraction", "number", "%", "divide by 100", "percent strain converted only in the explicit preview"],
    ["f in Hz", "frequency_hz", "number", "Hz", "identity", "oscillation frequency, not angular frequency"],
    ["T in °C", "temperature_c", "number", "°C", "identity", "recorded temperature; not replaced by a nominal set point"],
    ["ɣ̇ in 1/s", "shear_rate_per_s", "number", "1/s", "identity", "steady-shear rate"],
    ["η in mPas", "apparent_viscosity_source", "number", "mPas", "deferred", "steady-shear apparent viscosity; source unit token retained"],
], columns=["source_header", "canonical_field", "logical_type", "source_unit", "conversion_policy", "definition"])
observed_headers = set(audit_tables["source_header_inventory.csv"]["source_header"])
require(set(field_mapping.source_header) == observed_headers, "Field map does not cover exactly the audited source headers.")
require(field_mapping.source_header.is_unique, "Source headers must have a unique mapping.")
image_mapping = pd.DataFrame([
    ["image_id", "string", "Source image label; preserve as text, including any leading zeros.", "none"],
    ["date", "string", "Source date text; validate calendar format without asserting the event it dates.", "none"],
    ["ink_type_ph", "string", "Source ink label; no concentration extraction or formulation foreign key yet.", "unresolved composition mapping"],
    ["shape", "string", "Source shape category; not a fidelity or performance score.", "none"],
    ["intensity", "string", "Source intensity token; literature supports a percentage-setting interpretation only provisionally.", "unresolved CSV unit definition"],
    ["completeness_sam", "string", "Source annotation; retain the literal token and unknown blanks.", "annotation semantics unresolved"],
    ["contain_sps", "string", "Source reagent annotation; blank does not establish absence or presence.", "annotation semantics unresolved"],
], columns=["source_column", "logical_type", "definition", "semantic_limit"])
display(field_mapping)
display(image_mapping)


,source_header,canonical_field,logical_type,source_unit,conversion_policy,definition
0,SegIndex,segment_index_source,string,not applicable,retain,source acquisition label; detailed segmentation semantics unresolved
1,t in s,time_s,number,s,identity,source time coordinate
2,t_seg in s,segment_time_s,number,s,identity,source segment-time coordinate; zero-time convention unresolved
3,τ in Pa,shear_stress_pa,number,Pa,identity,stress under the recorded test mode
4,G' in Pa,storage_modulus_pa,number,Pa,identity,elastic storage contribution under recorded oscillation conditions
5,"G"" in Pa",loss_modulus_pa,number,Pa,identity,viscous loss contribution under recorded oscillation conditions
6,|η*| in mPas,complex_viscosity_source,number,mPas,deferred,oscillatory complex viscosity; source unit token retained
7,ɣ in -,strain_fraction,number,-,identity,dimensionless strain fraction as labelled
8,ɣ in %,strain_fraction,number,%,divide by 100,percent strain converted only in the explicit preview
9,f in Hz,frequency_hz,number,Hz,identity,"oscillation frequency, not angular frequency"


,source_column,logical_type,definition,semantic_limit
0,image_id,string,"Source image label; preserve as text, including any leading zeros.",none
1,date,string,Source date text; validate calendar format without asserting the event it dates.,none
2,ink_type_ph,string,Source ink label; no concentration extraction or formulation foreign key yet.,unresolved composition mapping
3,shape,string,Source shape category; not a fidelity or performance score.,none
4,intensity,string,Source intensity token; literature supports a percentage-setting interpretation only provisionally.,unresolved CSV unit definition
5,completeness_sam,string,Source annotation; retain the literal token and unknown blanks.,annotation semantics unresolved
6,contain_sps,string,Source reagent annotation; blank does not establish absence or presence.,annotation semantics unresolved


**Observed output.** The map covers all 13 observed rheology headers and all seven image-index fields. Two strain headers map to a common proposed fraction representation with separate policies. Both viscosity fields retain the source unit token `mPas`; no viscosity conversion has been approved or applied.


## 6. Instantiate the source-preserving schema

**Question.** Can every audited worksheet, point and cell be represented without losing source coordinates?

**Data-science rationale.** This is a bounded schema-validation prototype using the actual workbooks. It reads archive members in place, preserves source values as JSON scalars, and records original row and column coordinates. JSON distinguishes a number from a numeric-looking string; `null` represents an empty source cell. A long cell table is easier to trace across the seven-column and ten-column worksheet layouts than a wide table filled with artificial null columns.

The archive member hash, sheet name, headers and row count must agree with Phase 1. Formula and Excel-error cells are rejected because this prototype is not a spreadsheet calculation engine. Non-numeric or nonfinite measurements remain represented and receive a parsing status rather than being dropped. The numeric interpretation is an additional field, never a replacement for the source scalar.

**Materials rationale.** Each measurement row keeps its recorded stress, deformation or shear rate, temperature and time together through a common point key. The folder and headers support the broad labels `oscillatory_moduli` and `steady_shear_viscosity`; these labels do not establish a complete specimen history. The source basename is retained without interpreting a suffix as a replicate number.

**Output and boundary.** The three tables prove source coverage at the declared granularity. They do not turn the worksheet register into a physical-sample register. No measurement value is corrected, averaged, interpolated or removed for being unusual. Completely blank rows are excluded as non-measurement rows only under the audited count contract.


In [18]:
map_by_header = field_mapping.set_index("source_header").to_dict("index")
archive_inventory = audit_tables["archive_member_inventory.csv"]
member_hashes = dict(zip(archive_inventory.member_path, archive_inventory.sha256))
worksheet_rows, point_rows, value_rows = [], [], []
with zipfile.ZipFile(zip_path) as archive:
    require(len(archive.namelist()) == len(set(archive.namelist())), "Duplicate archive member names.")
    expected_members = set(workbook_inventory.member_path)
    actual_members = {n for n in archive.namelist() if n.lower().endswith(".xlsx")}
    require(actual_members == expected_members, "Workbook coverage differs from Phase 1.")
    for member, group in workbook_inventory.groupby("member_path", sort=True):
        payload = archive.read(member)
        require(hashlib.sha256(payload).hexdigest() == member_hashes[member], f"Member checksum mismatch: {member}")
        wb = openpyxl.load_workbook(io.BytesIO(payload), read_only=True, data_only=False)
        try:
            require(set(wb.sheetnames) == set(group.sheet), f"Sheet coverage changed: {member}")
            for expected in group.itertuples(index=False):
                ws = wb[expected.sheet]
                rows = list(ws.iter_rows())
                header_row = int(expected.candidate_header_row)
                require(header_row == 1, "This checkpoint expects headers in Excel row 1.")
                headers = ["" if c.value is None else str(c.value) for c in rows[0]]
                require(headers == json.loads(expected.headers_json), f"Header mismatch: {member} / {ws.title}")
                require(len(headers) == len(set(headers)) and all(headers), "Blank or duplicate headers.")
                require(all(c.data_type not in {"f", "e"} for row in rows for c in row), "Formula or Excel-error cell requires review.")
                active = [(i, row) for i, row in enumerate(rows[1:], 2) if not all(is_blank(c.value) for c in row)]
                require(len(active) == int(expected.data_rows), f"Point-count mismatch: {member}")
                require(len(rows)-1-len(active) == int(expected.blank_data_rows), "Blank-row count changed.")
                family = ("oscillatory_moduli" if {"G' in Pa", 'G" in Pa', "f in Hz"}.issubset(headers)
                          else "steady_shear_viscosity" if {"η in mPas", "ɣ̇ in 1/s"}.issubset(headers) else "unresolved")
                worksheet_rows.append(dict(member_path=member, sheet=ws.title,
                    source_stem=PurePosixPath(member).stem, member_sha256=member_hashes[member],
                    measurement_family=family, header_row=header_row, point_count=len(active),
                    physical_sample_link_status="unresolved"))
                for excel_row, row in active:
                    point_rows.append(dict(member_path=member, sheet=ws.title, source_excel_row=excel_row))
                    for position, (header, cell) in enumerate(zip(headers, row), 1):
                        value = cell.value
                        source_json = json.dumps(value, ensure_ascii=False, allow_nan=False)
                        if is_blank(value):
                            numeric, status = np.nan, "blank_unknown"
                        elif map_by_header[header]["logical_type"] == "string":
                            numeric, status = np.nan, "text_preserved"
                        else:
                            try:
                                numeric = float(value)
                                status = "numeric" if np.isfinite(numeric) else "nonfinite"
                                if isinstance(value, bool):
                                    status = "invalid_boolean"
                            except (ValueError, TypeError):
                                numeric, status = np.nan, "nonnumeric"
                        value_rows.append(dict(member_path=member, sheet=ws.title, source_excel_row=excel_row,
                            source_column_position=position, source_header=header, source_value_json=source_json,
                            source_unit=map_by_header[header]["source_unit"], numeric_value=numeric,
                            parse_status=status))
        finally:
            wb.close()
worksheet_register = pd.DataFrame(worksheet_rows)
measurement_points = pd.DataFrame(point_rows)
measurement_values = pd.DataFrame(value_rows)
display(worksheet_register.groupby("measurement_family").agg(worksheets=("sheet", "size"), points=("point_count", "sum")).reset_index())
display(measurement_values.head(10))
print("Source cells represented:", len(measurement_values))


,measurement_family,worksheets,points
0,oscillatory_moduli,46,2300
1,steady_shear_viscosity,46,2300


,member_path,sheet,source_excel_row,source_column_position,source_header,source_value_json,source_unit,numeric_value,parse_status
0,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,1,SegIndex,"""1|1""",not applicable,NaN,text_preserved
1,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,2,t in s,6.45058584213257,s,6.450586,numeric
2,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,3,t_seg in s,5.87934732437134,s,5.879347,numeric
3,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,4,τ in Pa,0.000957517419010401,Pa,0.000958,numeric
4,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,5,G' in Pa,0.0919053554534912,Pa,0.091905,numeric
5,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,6,"G"" in Pa",0.0349451825022697,Pa,0.034945,numeric
6,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,7,|η*| in mPas,15.6488716602325,mPas,15.648872,numeric
7,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,8,ɣ in -,0.00973831303417683,-,0.009738,numeric
8,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,9,f in Hz,1,Hz,1.000000,numeric
9,GG_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0,2,10,T in °C,22.5002380371094,°C,22.500238,numeric


Source cells represented: 39100


**Observed output.** All 92 worksheets were represented: 46 oscillatory-modulus records and 46 steady-shear-viscosity records, each family contributing 2,300 points. The source-cell table contains 39,100 cells: 4,600 acquisition labels and 34,500 numeric measurement or condition values. The point register contains 4,600 rows. This demonstrates extraction coverage, not independent replication.


## 7. Image metadata and missing-value semantics

**Question.** Can the source index be retained without converting missing annotations into experimental claims?

**Data-science rationale.** CSV values are read as literal strings. The schema adds the CSV record number and a separate status for each source field. A blank or whitespace-only value receives `unknown_blank`; all other tokens remain `reported`. This is an observed missingness classification, not an explanation of why a measurement or annotation is absent. We cannot distinguish “not measured”, “not applicable” and “not recorded” without evidence.

The expected filename is taken directly from Phase 1 and linked using both record number and source image ID. It is only a naming expectation: the images were not supplied. Observed categories form a snapshot vocabulary used to detect drift; they are not a universal list of allowed future categories.

**Polymer-science rationale.** An unfilled `contain_sps` field does not establish whether the ink contained the crosslinking reagent. Likewise, a blank `completeness_sam` annotation cannot be converted into successful printing. Such changes would create false process or performance labels. An ink label alone cannot establish molecular composition or the identity of the specimen used in rheology.

**Output and boundary.** Image rows remain a separate table. Even unique image IDs do not establish independent constructs or runs. No image-quality metric is generated and no rheology-to-image join is performed.


In [21]:
with image_path.open(encoding="utf-8-sig", newline="") as handle:
    image_csv = list(csv.reader(handle))
image_headers = image_csv[0]
require(image_headers == image_mapping.source_column.tolist(), "Image-index headers differ from the schema.")
require(all(len(r) == len(image_headers) for r in image_csv[1:]), "Malformed CSV record width.")
image_records = pd.DataFrame(image_csv[1:], columns=image_headers)
image_records.insert(0, "source_csv_record", range(2, len(image_records)+2))
for column in image_headers:
    image_records[column + "_status"] = np.where(image_records[column].str.strip().eq(""), "unknown_blank", "reported")
expectations = audit_tables["image_filename_expectations.csv"].copy()
expectations["source_csv_record"] = expectations.source_csv_record.astype(int)
image_records = image_records.merge(expectations[["source_csv_record", "image_id", "expected_filename", "existence_status"]],
                                    on=["source_csv_record", "image_id"], how="left", validate="one_to_one")
require(image_records.expected_filename.notna().all(), "An image-index record lacks its Phase 1 filename expectation.")
image_missingness = pd.DataFrame([dict(source_column=c, rows=len(image_records),
    unknown_blank=int(image_records[c + "_status"].eq("unknown_blank").sum())) for c in image_headers])
observed_categories = audit_tables["image_index_value_counts.csv"].copy()
missing_value_rules = pd.DataFrame([
    ["blank source cell / CSV token", "unknown", "Preserve the original scalar/token and mark observed blankness."],
    ["literal no", "reported source annotation", "Retain no; do not infer the annotation's full scope or calibration."],
    ["nonnumeric measurement", "unparseable", "Retain the source scalar and flag; do not impute."],
    ["quantity absent from a worksheet layout", "structurally absent", "No source-cell row is invented for an absent header."],
    ["unresolved unit or relationship", "unresolved semantics", "Keep the value or label; do not create a converted value or scientific foreign key."],
], columns=["source_condition", "interpretation", "action"])
display(image_missingness)
display(missing_value_rules)


,source_column,rows,unknown_blank
0,image_id,1031,0
1,date,1031,0
2,ink_type_ph,1031,0
3,shape,1031,0
4,intensity,1031,0
5,completeness_sam,1031,943
6,contain_sps,1031,922


,source_condition,interpretation,action
0,blank source cell / CSV token,unknown,Preserve the original scalar/token and mark observed blankness.
1,literal no,reported source annotation,Retain no; do not infer the annotation's full scope or calibration.
2,nonnumeric measurement,unparseable,Retain the source scalar and flag; do not impute.
3,quantity absent from a worksheet layout,structurally absent,No source-cell row is invented for an absent header.
4,unresolved unit or relationship,unresolved semantics,Keep the value or label; do not create a converted value or scientific foreign key.


**Observed output.** All 1,031 index records were retained. `completeness_sam` has 943 blanks and `contain_sps` has 922 blanks; each blank remains unknown. The other five source fields are populated in this snapshot. The nonblank values in the two annotation fields are the literal token `no`; an unannotated row is not thereby a `yes`.


## 8. The data dictionary as an executable contract

**Question.** Is every field in the instantiated tables explained?

**Data-science rationale.** A data dictionary defines field meanings, types, units, provenance and missingness. It is checked against the actual table columns rather than maintained as an unrelated document. This reduces the risk that new fields enter the workflow without explanation. Composite keys and joins are tested next; a dictionary alone cannot enforce them.

**Materials rationale.** A distinction between a reported scalar and its numeric interpretation protects scientific meaning. Likewise, “physical sample link unresolved” is a status of knowledge, not a material property. The dictionary separates technical coordinates from measurements so they are not accidentally used as chemical descriptors.

**Output and boundary.** All fields of the four core prototype tables must be covered. Definitions for the source rheology quantities and image fields are supplied by the source mappings above; auxiliary evidence and QC tables describe decisions rather than additional experimental entities.


In [24]:
technical_definitions = {
 "member_path": ("string", "Exact workbook path inside the verified ZIP; worksheet-key component."),
 "sheet": ("string", "Exact source worksheet name; worksheet-key component."),
 "source_stem": ("string", "Unparsed workbook basename without extension; not a sample ID."),
 "member_sha256": ("string", "SHA-256 of workbook bytes from the verified archive."),
 "measurement_family": ("string", "Header-supported measurement family; not a complete protocol or specimen state."),
 "header_row": ("integer", "One-based source Excel header row."),
 "point_count": ("integer", "Count of nonblank source data rows; not independent sample size."),
 "physical_sample_link_status": ("string", "Unresolved physical identity; no physical-sample foreign key exists."),
 "source_excel_row": ("integer", "One-based original Excel row coordinate; point-key component."),
 "source_column_position": ("integer", "One-based original Excel column coordinate; cell-key component."),
 "source_header": ("string", "Exact source column header; foreign key to field_mapping.source_header."),
 "source_value_json": ("string", "JSON-encoded source scalar; retains number/text/null distinction."),
 "source_unit": ("string", "Exact header unit token, or not applicable for a label."),
 "numeric_value": ("number or null", "Parsed source numeric value with no unit conversion; consult parse_status."),
 "parse_status": ("string", "numeric, text_preserved, blank_unknown, nonnumeric, nonfinite or invalid_boolean."),
 "source_csv_record": ("integer", "One-based CSV record number, counting header as record 1; not a physical line counter."),
 "expected_filename": ("string", "Phase 1 filename expectation; file existence is not asserted."),
 "existence_status": ("string", "Phase 1 image-existence status carried forward verbatim."),
}
core_tables = {"worksheet_register": worksheet_register, "measurement_points": measurement_points,
               "measurement_values": measurement_values, "image_records": image_records}
dictionary_rows = []
for table, frame in core_tables.items():
    for field in frame.columns:
        if table == "image_records" and field in image_headers:
            description = image_mapping.set_index("source_column").loc[field, "definition"]
            dtype, origin = "string", "source-reported token"
        elif table == "image_records" and field.endswith("_status"):
            dtype, description, origin = "string", "reported or unknown_blank for " + field[:-7], "derived missingness status"
        else:
            require(field in technical_definitions, f"Undocumented field: {table}.{field}")
            dtype, description = technical_definitions[field]
            origin = "source locator, representation or explicitly derived metadata"
        dictionary_rows.append(dict(table=table, field=field, logical_type=dtype, definition=description,
            provenance=origin, unit_policy="See exact source_header mapping for measurement values; no implicit conversion.",
            missingness_policy="Source blanks retained; derived nulls require the associated parsing/status field."))
data_dictionary = pd.DataFrame(dictionary_rows)
require(not data_dictionary.duplicated(["table", "field"]).any(), "Duplicate dictionary definitions.")
display(data_dictionary[["table", "field", "logical_type", "definition"]])


,table,field,logical_type,definition
0,worksheet_register,member_path,string,Exact workbook path inside the verified ZIP; worksheet-key component.
1,worksheet_register,sheet,string,Exact source worksheet name; worksheet-key component.
2,worksheet_register,source_stem,string,Unparsed workbook basename without extension; not a sample ID.
3,worksheet_register,member_sha256,string,SHA-256 of workbook bytes from the verified archive.
4,worksheet_register,measurement_family,string,Header-supported measurement family; not a complete protocol or specimen state.
5,worksheet_register,header_row,integer,One-based source Excel header row.
6,worksheet_register,point_count,integer,Count of nonblank source data rows; not independent sample size.
7,worksheet_register,physical_sample_link_status,string,Unresolved physical identity; no physical-sample foreign key exists.
8,measurement_points,member_path,string,Exact workbook path inside the verified ZIP; worksheet-key component.
9,measurement_points,sheet,string,Exact source worksheet name; worksheet-key component.


**Observed output.** The dictionary defines all 37 fields across the four prototype tables. The fields describe source tokens, coordinates, measurements and explicit metadata statuses. No field creates a physical-sample identifier or asserts a printing-to-rheology relationship.


## 9. Quality-control rules and structural validation

**Question.** Do the implemented tables satisfy their keys, row counts and parent–child relationships?

**Data-science rationale.** Referential integrity means every child record has an existing parent. The `many_to_one` merge validation below prevents accidental multiplication of measurement rows. Uniqueness, cardinality, dictionary coverage and source reconciliation are deterministic checks. These are data-contract tests, not a statistical assessment of the experiments.

**Severity policy.** **ERROR** means an input or structural contract is broken and the notebook stops. **WARNING** means a record requires scientific or semantic review; it is retained. **INFO** documents a bounded status or transformation. A warning is not automatically an invalid experiment. Phase 1's original `review` labels remain unchanged in its carried-forward table.

**Materials rationale.** Accurate parent–child links preserve the conditions associated with each property. A false join can attach one ink's strain or temperature to another ink's modulus. However, a correct source-record relationship still cannot prove that two worksheets describe the same specimen. Table integrity and experimental independence are separate questions.

**Output and boundary.** The summary records what was tested and whether the contract passed. A passed summary means the tables preserve the audited source structure; it does not mean the dataset is ready for inference or prediction.


In [27]:
qc_rules = pd.DataFrame([
 ["CHECKPOINT_HASH", "ERROR", "Input and audit hashes must match the reviewed snapshot.", "Stop on mismatch."],
 ["SOURCE_COVERAGE", "ERROR", "Every audited workbook, sheet, point and cell must be represented.", "Stop on missing or additional records."],
 ["PRIMARY_KEY", "ERROR", "Natural source-coordinate keys must be unique and nonblank.", "Stop; do not deduplicate silently."],
 ["FOREIGN_KEY", "ERROR", "Every child record must have exactly one source parent.", "Stop on orphan or multiplication."],
 ["NUMERIC_PARSE", "WARNING", "Every measurement value should be finite numeric or explicitly flagged.", "Keep original scalar and parse status."],
 ["NONPOSITIVE_RESPONSE", "WARNING", "Viscosity and moduli <= 0 need review before logs or fitting.", "Retain; cause is not established."],
 ["ACQUISITION_AXIS", "WARNING", "Nonpositive frequency/shear rate or negative strain needs review.", "Retain and inspect test context."],
 ["DUPLICATE_ROW", "WARNING", "Identical full rows in a worksheet need contextual review.", "Retain; do not assume an export error."],
 ["IMAGE_ID", "WARNING", "Blank or repeated image labels need review.", "Retain source record coordinates."],
 ["IMAGE_DATE", "WARNING", "Source dates should parse as YYYY/MM/DD calendar dates.", "Retain original text."],
 ["IMAGE_BLANK", "WARNING", "Blank image metadata remains unknown.", "Do not fill with zero, yes or no."],
 ["IMAGE_VOCABULARY", "ERROR", "Categories and counts must match the Phase 1 snapshot.", "Stop on unexplained drift."],
 ["UNRESOLVED_SEMANTICS", "WARNING", "Unit, annotation and physical-link uncertainties remain explicit.", "Defer unsupported transformations and joins."],
 ["STRAIN_PREVIEW", "INFO", "Percent-to-fraction arithmetic must be reversible within floating-point tolerance.", "Preview only; keep the source scalar."],
], columns=["rule", "severity", "criterion", "action"])
checks = []
def check(name, passed, evidence):
    checks.append(dict(check=name, status="PASS" if bool(passed) else "FAIL", evidence=str(evidence)))
    require(passed, f"Structural check failed: {name}. {evidence}")
worksheet_key = ["member_path", "sheet"]
point_key = worksheet_key + ["source_excel_row"]
cell_key = point_key + ["source_column_position"]
for name, frame, key in [("worksheet", worksheet_register, worksheet_key),
                         ("point", measurement_points, point_key),
                         ("cell", measurement_values, cell_key),
                         ("image record", image_records, ["source_csv_record"])]:
    check(name + " primary key", not frame.duplicated(key).any() and not frame[key].isna().any().any()
          and frame[key].astype(str).apply(lambda s: s.str.strip().ne("")).all().all(), len(frame))
for name, child, parent, key in [("point to worksheet", measurement_points, worksheet_register, worksheet_key),
                                 ("cell to point", measurement_values, measurement_points, point_key)]:
    linked = child[key].merge(parent[key], on=key, how="left", validate="many_to_one", indicator=True)
    check(name, linked._merge.eq("both").all() and len(linked) == len(child), "No orphan or multiplied records")
check("worksheet count", len(worksheet_register) == phase1_manifest["worksheets_audited"], len(worksheet_register))
check("point count", len(measurement_points) == phase1_manifest["measurement_rows"], len(measurement_points))
check("image count", len(image_records) == phase1_manifest["image_index_rows"], len(image_records))
check("cell count", len(measurement_values) == column_profile.observed_rows.astype(int).sum(), len(measurement_values))
cell_counts = measurement_values.groupby(worksheet_key + ["source_column_position"]).size().rename("actual").reset_index()
expected_counts = column_profile[["member_path", "sheet", "column_position", "observed_rows"]].copy()
expected_counts["source_column_position"] = expected_counts.pop("column_position").astype(int)
count_comparison = cell_counts.merge(expected_counts, on=worksheet_key+["source_column_position"], how="outer", validate="one_to_one")
check("per-column coverage", count_comparison.actual.eq(count_comparison.observed_rows.astype(int)).all(), len(count_comparison))
check("dictionary coverage", len(data_dictionary) == sum(len(t.columns) for t in core_tables.values()), len(data_dictionary))
check("source-header mapping", measurement_values.source_header.isin(field_mapping.source_header).all(), "All source headers mapped")
display(qc_rules)
display(pd.DataFrame(checks))


,rule,severity,criterion,action
0,CHECKPOINT_HASH,ERROR,Input and audit hashes must match the reviewed snapshot.,Stop on mismatch.
1,SOURCE_COVERAGE,ERROR,"Every audited workbook, sheet, point and cell must be represented.",Stop on missing or additional records.
2,PRIMARY_KEY,ERROR,Natural source-coordinate keys must be unique and nonblank.,Stop; do not deduplicate silently.
3,FOREIGN_KEY,ERROR,Every child record must have exactly one source parent.,Stop on orphan or multiplication.
4,NUMERIC_PARSE,WARNING,Every measurement value should be finite numeric or explicitly flagged.,Keep original scalar and parse status.
5,NONPOSITIVE_RESPONSE,WARNING,Viscosity and moduli <= 0 need review before logs or fitting.,Retain; cause is not established.
6,ACQUISITION_AXIS,WARNING,Nonpositive frequency/shear rate or negative strain needs review.,Retain and inspect test context.
7,DUPLICATE_ROW,WARNING,Identical full rows in a worksheet need contextual review.,Retain; do not assume an export error.
8,IMAGE_ID,WARNING,Blank or repeated image labels need review.,Retain source record coordinates.
9,IMAGE_DATE,WARNING,Source dates should parse as YYYY/MM/DD calendar dates.,Retain original text.


,check,status,evidence
0,worksheet primary key,PASS,92
1,point primary key,PASS,4600
2,cell primary key,PASS,39100
3,image record primary key,PASS,1031
4,point to worksheet,PASS,No orphan or multiplied records
5,cell to point,PASS,No orphan or multiplied records
6,worksheet count,PASS,92
7,point count,PASS,4600
8,image count,PASS,1031
9,cell count,PASS,39100


**Observed output.** All 13 structural checks in this section passed, including primary keys, parent–child links, counts and coverage of 782 source columns. Fourteen QC rules were documented. Passing these checks establishes a valid representation of the snapshot; scientific comparability remains a separate assessment.


## 10. Value-level QC without silent cleaning

**Question.** Which source values and annotations need review after the structure has been validated?

**Data-science rationale.** A QC finding needs a rule, source location, evidence and an action. We preserve cell coordinates for measurement findings and CSV record numbers for image findings. Missing image annotations are reported by column with their counts; the per-row status columns preserve their exact locations. Counts of findings, affected cells, curves and independent samples must not be confused.

The prototype reconciles detected nonpositive rheological response cells with the exact locations in the Phase 1 issue log. It also tests numeric parsing, acquisition axes, duplicate full source rows, image identifiers, dates and the observed category counts. The vocabulary check uses all Phase 1 category counts, including blanks, rather than assuming that a plausible label is valid.

**Rheology rationale.** A nonpositive viscosity value is a review signal before applying a logarithm or fitting a constitutive curve. It is not, on its own, evidence that the material has negative physical viscosity. Instrument sensitivity, signal processing and the experimental regime are possible explanations, but none is established here. The value is retained. Arbitrary “reasonable” temperature or modulus ranges are not used to delete data.

**Output and boundary.** QC produces review records, not repaired measurements. Blank annotations remain unknown, and no questionable curve point is removed to improve a plot or model score. A future analysis must state its eligibility rules and examine sensitivity to unresolved values.


In [30]:
findings = []
def finding(rule, source, location, evidence, action, severity="WARNING"):
    findings.append(dict(rule=rule, severity=severity, source=source, location=location,
                         evidence=str(evidence), action=action))
responses = {"G' in Pa", 'G" in Pa', "η in mPas", "|η*| in mPas"}
for row in measurement_values.itertuples(index=False):
    loc = f"{row.sheet}; row {row.source_excel_row}; column {row.source_column_position}; {row.source_header}"
    if row.parse_status not in {"numeric", "text_preserved"}:
        finding("NUMERIC_PARSE", row.member_path, loc, row.source_value_json, "Retain source scalar; review type or missingness.")
    if row.parse_status == "numeric":
        if row.source_header in responses and row.numeric_value <= 0:
            finding("NONPOSITIVE_RESPONSE", row.member_path, loc, row.numeric_value, "Retain; assess measurement limits before modelling.")
        if ((row.source_header in {"ɣ̇ in 1/s", "f in Hz"} and row.numeric_value <= 0) or
            (row.source_header in {"ɣ in -", "ɣ in %"} and row.numeric_value < 0)):
            finding("ACQUISITION_AXIS", row.member_path, loc, row.numeric_value, "Retain and review acquisition context.")
wide_raw = measurement_values.pivot(index=point_key, columns="source_column_position", values="source_value_json")
for key, frame in wide_raw.groupby(level=[0, 1], sort=True):
    for duplicate in frame[frame.duplicated(keep=False)].index:
        finding("DUPLICATE_ROW", key[0], f"{key[1]}; row {duplicate[2]}", "Identical full source row", "Retain pending contextual review.")
for row in image_missingness.itertuples(index=False):
    if row.unknown_blank:
        finding("IMAGE_BLANK", "image_index.csv", row.source_column, f"{row.unknown_blank}/{row.rows} unknown blanks", "Keep unknown; do not infer annotation outcomes.")
bad_ids = image_records.image_id.str.strip().eq("") | image_records.image_id.duplicated(keep=False)
for row in image_records.loc[bad_ids].itertuples(index=False):
    finding("IMAGE_ID", "image_index.csv", f"record {row.source_csv_record}", row.image_id, "Retain; investigate source image identity.")
valid_date_shape = image_records.date.str.fullmatch(r"\d{4}/\d{2}/\d{2}")
parsed_dates = pd.to_datetime(image_records.date, format="%Y/%m/%d", errors="coerce")
for row in image_records.loc[~valid_date_shape | parsed_dates.isna()].itertuples(index=False):
    finding("IMAGE_DATE", "image_index.csv", f"record {row.source_csv_record}", row.date, "Retain original date; review format.")
for column in observed_categories.column.unique():
    expected = observed_categories.loc[observed_categories.column.eq(column)]
    expected_counts = {r.source_value: int(r.count) for r in expected.itertuples(index=False)}
    actual_counts = image_records[column].value_counts(dropna=False).to_dict()
    check("category counts: " + column, actual_counts == expected_counts, sum(actual_counts.values()))
for row in audit_tables["image_index_column_profile.csv"].itertuples(index=False):
    actual_blank = int(image_records[row.column + "_status"].eq("unknown_blank").sum())
    check("image missingness: " + row.column, actual_blank == int(row.blank_count), actual_blank)
phase1_nonpositive = set()
for row in phase1_issues.loc[phase1_issues.issue_code.eq("NONPOSITIVE_RHEOLOGY_VALUE")].itertuples(index=False):
    for excel_row in json.loads(row.evidence.split(":", 1)[1].strip()):
        phase1_nonpositive.add((row.source, row.sheet, excel_row, row.column))
nonpositive_cells = measurement_values.loc[measurement_values.source_header.isin(responses)
    & measurement_values.parse_status.eq("numeric") & measurement_values.numeric_value.le(0)].copy()
actual_nonpositive = set(nonpositive_cells[["member_path", "sheet", "source_excel_row", "source_header"]].itertuples(index=False, name=None))
check("nonpositive response locations agree with Phase 1", actual_nonpositive == phase1_nonpositive, len(actual_nonpositive))
value_qc_findings = pd.DataFrame(findings, columns=["rule", "severity", "source", "location", "evidence", "action"])
display(value_qc_findings.groupby(["rule", "severity"]).size().rename("finding_count").reset_index())
display(value_qc_findings.head(12))
print("Nonpositive response cells:", len(nonpositive_cells), "; affected worksheets:",
      len(nonpositive_cells[worksheet_key].drop_duplicates()))


,rule,severity,finding_count
0,ACQUISITION_AXIS,WARNING,2
1,IMAGE_BLANK,WARNING,2
2,NONPOSITIVE_RESPONSE,WARNING,20


,rule,severity,source,location,evidence,action
0,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 2; column 4; η in mPas,-501.090288162231,Retain; assess measurement limits before modelling.
1,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 3; column 4; η in mPas,-85.0892066955566,Retain; assess measurement limits before modelling.
2,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 4; column 4; η in mPas,-98.492406308651,Retain; assess measurement limits before modelling.
3,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 5; column 4; η in mPas,-168.703719973564,Retain; assess measurement limits before modelling.
4,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 6; column 4; η in mPas,-150.822579860687,Retain; assess measurement limits before modelling.
5,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 7; column 4; η in mPas,-38.880068808794,Retain; assess measurement limits before modelling.
6,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.25_HA-HQ-Ph_0.5_22.5C_0.xlsx,ALG_I1G_Ph_0____Ph_0_5_22_5C_0; row 8; column 4; η in mPas,-50.4644736647606,Retain; assess measurement limits before modelling.
7,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0; row 2; column 4; η in mPas,-791.833639144897,Retain; assess measurement limits before modelling.
8,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0; row 3; column 4; η in mPas,-437.784135341644,Retain; assess measurement limits before modelling.
9,NONPOSITIVE_RESPONSE,WARNING,viscosity_data/ALG-I1G-Ph_0.75_HA-HQ-Ph_0.75_22.5C_0.xlsx,ALG_I1G_Ph_0___Ph_0_75_22_5C_0; row 4; column 4; η in mPas,-293.270945549011,Retain; assess measurement limits before modelling.


Nonpositive response cells: 20 ; affected worksheets: 8


**Observed output.** Value QC produced 24 finding records: 20 nonpositive response cells, two nonpositive shear-rate cells, and two column-level missing-annotation findings. The 20 response cells are viscosity values across eight worksheets, and their exact locations agree with Phase 1. Both flagged shear rates are negative. There were no numeric-parsing, duplicate-full-row, image-ID or date findings in this run. Category counts and missingness matched Phase 1. No flagged value was changed or removed.


## 11. Traceable strain-conversion preview

**Question.** Does a proposed percent-to-fraction transformation preserve both the source and its lineage?

**Data-science rationale.** A transformation record should retain the original scalar, original unit, numerical input, result, target unit and rule. The preview uses a separate table so source measurements cannot be overwritten. It checks round-trip arithmetic: converting a percentage to a fraction and back should recover the source number within a small floating-point tolerance. The tolerance concerns numerical representation only, not experimental precision.

**Rheology rationale.** Strain is the relative deformation in oscillatory testing. A percentage and a fraction represent the same deformation on different numerical scales. The conversion is `strain_fraction = strain_percent / 100`. A value already labelled `-` is retained numerically as a fraction. Large reported strains are not clipped to a plausible interval. Whether the source instrument's strain labelling is scientifically consistent and which values lie in the LVR remain separate questions.

**Output and boundary.** This preview validates arithmetic for the two documented representations. It does not choose a modulus plateau, calculate tan δ, or produce a fully standardised rheology dataset. Proposed viscosity conversion remains deferred; the preview is deliberately limited to supported strain representations.


In [33]:
strain_preview = measurement_values.loc[measurement_values.source_header.isin(["ɣ in -", "ɣ in %"])].copy()
require(strain_preview.parse_status.eq("numeric").all(), "Review strain parsing before generating a conversion preview.")
strain_preview["conversion_factor"] = np.where(strain_preview.source_unit.eq("%"), 0.01, 1.0)
strain_preview["standardised_value"] = strain_preview.numeric_value * strain_preview.conversion_factor
strain_preview["standardised_unit"] = "1 (strain fraction)"
strain_preview["conversion_rule"] = np.where(strain_preview.source_unit.eq("%"), "source value / 100", "identity: source fraction")
strain_preview["transformation_status"] = "schema preview; source unchanged"
check("strain conversion round trip", np.allclose(strain_preview.standardised_value / strain_preview.conversion_factor,
      strain_preview.numeric_value, rtol=1e-12, atol=0), len(strain_preview))
strain_summary = strain_preview.groupby("source_unit").agg(cells=("source_unit", "size"),
    source_min=("numeric_value", "min"), source_max=("numeric_value", "max"),
    fraction_min=("standardised_value", "min"), fraction_max=("standardised_value", "max")).reset_index()
display(strain_summary)
display(strain_preview.loc[strain_preview.source_unit.eq("%"),
    cell_key+["source_value_json", "source_unit", "standardised_value", "standardised_unit", "conversion_rule"]].head())


,source_unit,cells,source_min,source_max,fraction_min,fraction_max
0,%,100,0.005846,31.597120,0.000058,0.315971
1,-,2200,0.000071,335.218689,0.000071,335.218689


,member_path,sheet,source_excel_row,source_column_position,source_value_json,source_unit,standardised_value,standardised_unit,conversion_rule
15507,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,2,8,0.0113589638203848,%,0.000114,1 (strain fraction),source value / 100
15517,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,3,8,0.0120263954158872,%,0.000120,1 (strain fraction),source value / 100
15527,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,4,8,0.0123901612823829,%,0.000124,1 (strain fraction),source value / 100
15537,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,5,8,0.0123712059576064,%,0.000124,1 (strain fraction),source value / 100
15547,GG_data/ALG-I1G-Ph_4.0_HA-HQ-Ph_3.0_22.5C_0.xlsx,ALG_I1G_Ph_4____Ph_3_0_22_5C_0,6,8,0.0120190583402291,%,0.000120,1 (strain fraction),source value / 100


**Observed output.** The preview covers 2,300 strain cells: 2,200 labelled as fractions and 100 labelled as percentages. All round-trip arithmetic checks passed. The source-labelled fraction values extend to approximately 335.22, while the percent-derived fractions extend to approximately 0.316. These ranges must not be interpreted as a common LVR or corrected on the basis of plausibility alone.


## 12. Experimental context and comparability

**Question.** What do the existing acquisition-axis profiles establish about measurement conditions?

**Data-science rationale.** Comparability requires compatible definitions and conditions, not merely columns with the same units. We reuse Phase 1's axis profiles to show per-worksheet ranges and variation. These are descriptive diagnostics. A method statement and an observed range are retained side by side rather than forcing one to replace the other.

**Polymer and rheology rationale.** Moduli can change with strain amplitude, oscillation frequency and temperature. A constant frequency with varying strain is consistent with an amplitude sweep; it is not a frequency-dependent spectrum. The LVR is the regime in which the material's measured response is approximately independent of deformation amplitude under the relevant protocol. It must be evaluated from the curve and experimental context, not assigned from the column names.

The paper gives a nominal shear-rate range beginning at 0.01 s⁻¹. The workbook profile may contain points outside that range. Such a difference is preserved as a documentation-to-data review item; it is not a reason to delete observations. Similarly, measured temperature should not be replaced with the 22.5 °C nominal value, and no unreported instrument tolerance is invented.

**Output and boundary.** The context report describes available ranges and missing information. It does not fit curves, infer an LVR, establish material equivalence, or select representative values for each formulation. A future analysis must make those decisions explicitly and justify them.


In [36]:
axis_context = axis_profile.copy()
for c in ["numeric_min", "numeric_max", "distinct_numeric_values"]:
    axis_context[c] = pd.to_numeric(axis_context[c], errors="raise")
axis_context = axis_context.merge(worksheet_register[worksheet_key+["measurement_family"]],
                                  on=worksheet_key, how="left", validate="many_to_one")
frequency_context = axis_context.loc[axis_context.source_header.eq("f in Hz")]
shear_context = axis_context.loc[axis_context.source_header.eq("ɣ̇ in 1/s")]
context_summary = axis_context.groupby(["measurement_family", "source_header"]).agg(
    worksheets=("sheet", "size"), minimum=("numeric_min", "min"), maximum=("numeric_max", "max"),
    fewest_distinct_values=("distinct_numeric_values", "min"), most_distinct_values=("distinct_numeric_values", "max")).reset_index()
display(context_summary)
context_review = pd.DataFrame([
    {"question": "Oscillation frequency", "observed": f"{len(frequency_context)} sheets; {int((frequency_context.numeric_min.eq(1) & frequency_context.numeric_max.eq(1)).sum())} fixed at 1 Hz",
     "interpretation": "Consistent with the reported amplitude-sweep method; LVR still unassigned."},
    {"question": "Shear-rate lower bound", "observed": f"{int(shear_context.numeric_min.lt(0.01).sum())} worksheets include values below 0.01 1/s",
     "interpretation": "Retain those points and review the difference from the nominal range in Section 5.1."},
    {"question": "Independent sample count", "observed": "Not established by these source-coordinate keys",
     "interpretation": "Do not use worksheet or point counts as biological or physical replication."},
])
display(context_review)


,measurement_family,source_header,worksheets,minimum,maximum,fewest_distinct_values,most_distinct_values
0,oscillatory_moduli,T in °C,46,22.497675,22.503015,26,39
1,oscillatory_moduli,f in Hz,46,1.000000,1.000000,1,1
2,oscillatory_moduli,ɣ in %,2,0.005846,31.597120,50,50
3,oscillatory_moduli,ɣ in -,44,0.000071,335.218689,50,50
4,steady_shear_viscosity,T in °C,46,22.498224,22.502344,27,37
5,steady_shear_viscosity,ɣ̇ in 1/s,46,-0.000967,1000.525635,50,50


,question,observed,interpretation
0,Oscillation frequency,46 sheets; 46 fixed at 1 Hz,Consistent with the reported amplitude-sweep method; LVR still unassigned.
1,Shear-rate lower bound,46 worksheets include values below 0.01 1/s,Retain those points and review the difference from the nominal range in Section 5.1.
2,Independent sample count,Not established by these source-coordinate keys,Do not use worksheet or point counts as biological or physical replication.


**Observed output.** All 46 modulus worksheets report a fixed frequency of 1 Hz with varying strain, consistent with amplitude-sweep context. All 46 viscosity worksheets include a shear-rate value below the paper's nominal lower bound of 0.01 s⁻¹. Recorded temperatures span approximately 22.4977–22.5030 °C for the modulus files and 22.4982–22.5023 °C for the viscosity files. These ranges are descriptive; no acceptance tolerance or cause for the protocol-range difference is inferred.


## 13. Carry-forward decisions and linkage boundaries

**Question.** Which earlier issues have been clarified, and which remain unresolved?

**Data-science rationale.** Issue history should not disappear when a new phase starts. The original fifteen Phase 1 records are copied without modifying their evidence or severity. A separate decision table links to the existing issue IDs and documents Phase 2's response. Defining a rule is not the same as resolving the underlying scientific uncertainty.

Filename correspondences are retained as candidates only. Even an exact one-to-one filename match confirms a naming relationship, not a shared physical specimen. The reviewed correspondence table can quantify coverage without creating a scientific foreign key. Publication methods likewise cannot repair missing record-level keys.

**Materials rationale.** Separate aliquots, preparation batches, test histories and printing events can have different responses despite sharing formulation labels. Training and test splits would need experimentally justified grouping, not a random split of curve points. Image labels and worksheet names do not yet provide that grouping. No model-readiness claim follows from a successful schema validation.

**Output and boundary.** Mixed strain representations now have a documented preview rule, but the audit is not rewritten as if that variation never existed. The paper narrows the likely meaning of intensity without confirming the CSV convention or calibrating it to irradiance. Dataset reuse terms, image existence, annotations, viscosity-unit confirmation and physical linkage remain explicit review items.


In [39]:
phase1_issue_carry_forward = phase1_issues.copy(deep=True)
decision_by_code = {
 "DATASET_LICENCE_UNRESOLVED": ("open", "Dataset-specific reuse terms are not established by the article licence. No live licence reassessment in this notebook."),
 "IMAGE_FILES_NOT_AUDITED": ("open", "Expected filenames retained; no physical image files inspected."),
 "IMAGE_METADATA_BLANK": ("open; represented", "Explicit unknown_blank status introduced; no imputation or yes/no inference."),
 "INTENSITY_UNIT_UNRESOLVED": ("partly clarified; open", "Section 5.3 and Figure S3 support a percentage-setting interpretation; exact CSV semantics and calibration remain unverified."),
 "MIXED_STRAIN_UNITS": ("rule defined; preview tested", "Both source representations retained; fraction/percentage arithmetic is traceable. Full standardisation remains later work."),
 "NONPOSITIVE_RHEOLOGY_VALUE": ("open; locations reconciled", "Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion."),
 "PHYSICAL_SAMPLE_LINK_UNRESOLVED": ("open", "Source-coordinate keys identify records only; no sample or run identifiers created."),
}
require(set(phase1_issues.issue_code).issubset(decision_by_code), "A Phase 1 issue category has no Phase 2 decision.")
issue_decisions = pd.DataFrame([dict(phase1_issue_id=r.issue_id, issue_code=r.issue_code,
    phase2_status=decision_by_code[r.issue_code][0], decision=decision_by_code[r.issue_code][1])
    for r in phase1_issues.itertuples(index=False)])
check("issue carry-forward", phase1_issue_carry_forward.equals(phase1_issues), len(phase1_issue_carry_forward))
filename_candidates = audit_tables["candidate_filename_correspondences.csv"].copy()
linkage_summary = pd.DataFrame([
    {"quantity": "Source stems in candidate table", "count": len(filename_candidates)},
    {"quantity": "Stems with exactly one file in each family", "count": int(filename_candidates.exact_one_each.str.lower().eq("true").sum())},
    {"quantity": "Physical-sample links confirmed by this notebook", "count": 0},
])
additional_review = pd.DataFrame([
 ["VISCOSITY_UNIT_TOKEN", "open", "mPas retained; establish instrument unit notation before production conversion to Pa·s."],
 ["LVR_SELECTION", "open", "Study method reported; per-curve linear-regime selection and representative moduli not established."],
 ["PUBLICATION_RANGE", "open", "Observed shear-rate minima extend below the nominal method range; retain and review."],
 ["MODEL_POPULATION", "open", "Authors' 45 fitted-parameter data points are not reconciled to the 46 files per family; no forced exclusion."],
], columns=["review_topic", "status", "next_action"])
display(issue_decisions)
display(linkage_summary)
display(additional_review)


,phase1_issue_id,issue_code,phase2_status,decision
0,AUDIT-0001,DATASET_LICENCE_UNRESOLVED,open,Dataset-specific reuse terms are not established by the article licence. No live licence reassessment in this notebook.
1,AUDIT-0002,IMAGE_FILES_NOT_AUDITED,open,Expected filenames retained; no physical image files inspected.
2,AUDIT-0003,IMAGE_METADATA_BLANK,open; represented,Explicit unknown_blank status introduced; no imputation or yes/no inference.
3,AUDIT-0004,IMAGE_METADATA_BLANK,open; represented,Explicit unknown_blank status introduced; no imputation or yes/no inference.
4,AUDIT-0005,INTENSITY_UNIT_UNRESOLVED,partly clarified; open,Section 5.3 and Figure S3 support a percentage-setting interpretation; exact CSV semantics and calibration remain unverified.
5,AUDIT-0006,MIXED_STRAIN_UNITS,rule defined; preview tested,Both source representations retained; fraction/percentage arithmetic is traceable. Full standardisation remains later work.
6,AUDIT-0007,NONPOSITIVE_RHEOLOGY_VALUE,open; locations reconciled,Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion.
7,AUDIT-0008,NONPOSITIVE_RHEOLOGY_VALUE,open; locations reconciled,Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion.
8,AUDIT-0009,NONPOSITIVE_RHEOLOGY_VALUE,open; locations reconciled,Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion.
9,AUDIT-0010,NONPOSITIVE_RHEOLOGY_VALUE,open; locations reconciled,Affected cells matched to Phase 1 and retained unchanged; no causal diagnosis or exclusion.


,quantity,count
0,Source stems in candidate table,62
1,Stems with exactly one file in each family,30
2,Physical-sample links confirmed by this notebook,0


,review_topic,status,next_action
0,VISCOSITY_UNIT_TOKEN,open,mPas retained; establish instrument unit notation before production conversion to Pa·s.
1,LVR_SELECTION,open,Study method reported; per-curve linear-regime selection and representative moduli not established.
2,PUBLICATION_RANGE,open,Observed shear-rate minima extend below the nominal method range; retain and review.
3,MODEL_POPULATION,open,Authors' 45 fitted-parameter data points are not reconciled to the 46 files per family; no forced exclusion.


**Observed output.** All 15 Phase 1 findings were preserved and linked to explicit Phase 2 decisions. Four additional review topics were recorded. The candidate table contains 62 distinct source stems, of which 30 occur exactly once in each measurement family. No physical-sample link was confirmed. The next phase must retain that distinction when evaluating formulation or printing relationships.


## 14. Export the Phase 2 contract and handoff

**Question.** Can this work be regenerated, reviewed and passed to the next phase without altering its sources?

**Data-science rationale.** The exports contain the entity contract, dictionaries, prototype tables, source mappings, QC results and issue decisions. Each output receives a checksum in a run manifest. The manifest also records software versions and the verified input hashes. Generated tables use deterministic row order; a rerun replaces only the declared outputs. The final input hash comparison verifies preservation of both raw inputs and Phase 1 files.

**Materials rationale.** The handoff separates structural success from scientific readiness. It preserves exact source locations for follow-up on nonpositive responses, inconsistent units and uncertain relationships. It supplies evidence for later decisions about comparability and independence; it does not make those decisions by hiding missing context.

**Publication boundary.** The notebook code, schema definitions and reasoning are portfolio documentation. Source-cell and image tables contain third-party data, and their redistribution needs the dataset's own reuse terms to be established. This notebook exports them locally for review and does not upload or publish them. Do not automatically commit the raw ZIP, papers, working prompt or all generated tables to GitHub.

**Exit condition.** A complete run must preserve the checkpoint; cover every observed source field; pass source-coordinate, cardinality and coverage checks; retain Phase 1 issues; and leave unresolved scientific links visible. Warning-free data are not required. The next phase can implement validated standardisation and evidence-led linkage after resolving the relevant unit and identity questions. Materials Project, Matbench and GEMD remain separate future branches.


In [42]:
for path, original_hash in verified_paths.items():
    check("input preserved: " + path.name, sha256(path) == original_hash, "SHA-256 unchanged")
validation_summary = pd.DataFrame(checks)
exports = {
 "input_verification.csv": input_verification,
 "reviewed_inputs.csv": reviewed_inputs,
 "literature_evidence_register.csv": evidence_register,
 "entity_schema.csv": entity_schema,
 "deferred_entities.csv": deferred_entities,
 "source_to_schema_mapping.csv": field_mapping,
 "image_field_dictionary.csv": image_mapping,
 "data_dictionary.csv": data_dictionary,
 "missing_value_rules.csv": missing_value_rules,
 "worksheet_register.csv": worksheet_register,
 "measurement_points.csv": measurement_points,
 "measurement_values_source.csv": measurement_values,
 "image_records_source.csv": image_records,
 "image_missingness.csv": image_missingness,
 "image_observed_categories.csv": observed_categories,
 "quality_control_rules.csv": qc_rules,
 "validation_summary.csv": validation_summary,
 "value_qc_findings.csv": value_qc_findings,
 "strain_conversion_preview.csv": strain_preview,
 "measurement_context_summary.csv": context_summary,
 "context_review.csv": context_review,
 "phase1_issue_carry_forward.csv": phase1_issue_carry_forward,
 "phase2_issue_decisions.csv": issue_decisions,
 "additional_review_items.csv": additional_review,
 "filename_candidates_unconfirmed.csv": filename_candidates,
 "linkage_summary.csv": linkage_summary,
}
output_dir.mkdir(parents=True, exist_ok=True)
for name, table in exports.items():
    target = output_dir / name
    temporary = target.with_suffix(target.suffix + ".tmp")
    table.to_csv(temporary, index=False, encoding="utf-8-sig", lineterminator="\n")
    temporary.replace(target)
run_manifest = {
 "phase": "2: schema and QC prototype", "run_utc": datetime.now(timezone.utc).isoformat(),
 "source_record": phase1_manifest["source_record"], "source_version": "v0.0.2",
 "phase1_manifest_sha256": sha256(manifest_path),
 "environment": environment.to_dict("records"),
 "verified_input_sha256": {str(p.relative_to(project_root)): h for p, h in verified_paths.items()},
 "reviewed_supporting_inputs": reviewed_inputs.to_dict("records"),
 "rows": {name: len(table) for name, table in exports.items()},
 "export_sha256": {name: sha256(output_dir / name) for name in exports},
 "raw_and_phase1_files_preserved": True,
 "physical_sample_linkage": "unresolved", "image_existence": "not tested",
 "viscosity_unit_conversion": "deferred", "strain_conversion": "preview only",
 "dataset_licence": "unresolved at reviewed checkpoint", "hydrogel_model_readiness": "not established",
}
manifest_target = output_dir / "schema_qc_run_manifest.json"
temporary = manifest_target.with_suffix(".json.tmp")
temporary.write_text(json.dumps(run_manifest, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
temporary.replace(manifest_target)
display(pd.DataFrame([{"export": name, "rows": len(table)} for name, table in exports.items()]))
print("Structural checks passed:", len(validation_summary), "; value-QC finding records:", len(value_qc_findings))
print("Exports written to data/metadata/schema_and_quality_control/; all verified inputs unchanged.")


,export,rows
0,input_verification.csv,13
1,reviewed_inputs.csv,19
2,literature_evidence_register.csv,10
3,entity_schema.csv,4
4,deferred_entities.csv,4
5,source_to_schema_mapping.csv,13
6,image_field_dictionary.csv,7
7,data_dictionary.csv,37
8,missing_value_rules.csv,5
9,worksheet_register.csv,92


Structural checks passed: 43 ; value-QC finding records: 24
Exports written to data/metadata/schema_and_quality_control/; all verified inputs unchanged.


**Observed output.** All 43 recorded validation checks passed. The notebook wrote 26 CSV exports and one JSON run manifest to its dedicated output folder and confirmed that all 14 runtime input/checkpoint files remained byte-identical. The exports include unresolved findings and source-derived review tables; this is not a claim that the experimental data are fully standardised or model-ready.


## Interpretation of the verified run

The results below were recorded after executing and inspecting the complete notebook. All counts refer to the pinned source snapshot.

| Checkpoint | Verified result | Scientific interpretation |
|---|---:|---|
| Source worksheets | 92 | Digital measurement records, not confirmed independent samples |
| Measurement points | 4,600 | Repeated observations within those records |
| Source cells | 39,100 | Includes acquisition labels, conditions and measured responses |
| Image-index records | 1,031 | Metadata only; image-file existence not tested |
| Source fields mapped | 13 rheology headers; 7 image fields | Complete coverage of observed schemas |
| Core dictionary entries | 37 | All fields in the four instantiated tables documented |
| Structural and reconciliation checks | 43 passed | Source structure and lineage validated within the stated contract |
| Nonpositive response values | 20 cells in 8 worksheets | Retained; exact locations match Phase 1 |
| Nonpositive shear-rate values | 2 cells | Additional value-level QC flags; retained |
| Phase 1 findings carried forward | 15 | Original evidence and severity preserved |
| Confirmed physical-sample links | 0 | Filename matches remain candidate evidence |

The value-level QC table has 24 finding records. It combines cell-level findings with two column-level missingness summaries, so its row count is **not** a count of affected experiments. The eight Phase 1 nonpositive-response findings likewise describe groups of source locations, not eight individual values.

The paper reports 45 observations per fitted rheological parameter in its modelling dataset, while the source archive has 46 files per measurement family. This notebook records the difference without inventing exclusions or claiming to reproduce the authors' modelling population.

### What remains scientifically unresolved

The current tables preserve source identity and measurement context, but do not establish physical sample identities, a validated composition-to-measurement relationship, independent experimental replication, calibrated image-index intensity, per-curve LVR selection, or a justified hydrogel prediction target. Those limitations are part of the result, not shortcomings to conceal with additional columns.

A later linkage step should document the proposed key, its evidence, cardinality, unmatched records and the physical claim being made. A later rheology analysis should state the test conditions, retain individual curves, define valid transformations and distinguish curve points from independent materials. No authors' fitted models, generated images or performance results are represented as work produced here.

### References and project inputs

- Zhang, C., Elvitigala, K. C. M. L. and Sakai, S. (2026). Main publication, Sections 5.1–5.4 and associated supplementary Figures S1–S15. [Publication DOI](https://doi.org/10.1080/17452759.2026.2671497).
- [Zenodo dataset record 19602891](https://zenodo.org/records/19602891), reviewed Phase 1 snapshot `v0.0.2`; local source hashes verified in this notebook.
- Completed `01_zenodo_source_audit.ipynb` and eleven CSV outputs plus its JSON manifest. The reviewed filenames and hashes are recorded in `reviewed_inputs`.
- Project Phase 1 README and Hydrogel Bioink working prompt, supplied with this checkpoint. Their role is scope and method, not experimental measurement evidence.

This notebook uses the supplied source snapshot and documents the reviewed literature; it does not claim a live reassessment of the dataset version, reuse terms or external software APIs.
